In [2]:
# ==========================================
# PHASE 5.5 - CLV WITH CUSTOMER SEGMENTS
# ==========================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

# ------------------------------------------
# Load Datasets
# ------------------------------------------

clv_df = pd.read_csv(
    "../data/future_clv_dataset.csv"
)

segments = pd.read_csv(
    "../data/customer_segments.csv"
)

# ------------------------------------------
# Keep Required Columns
# ------------------------------------------

segments = segments[
    ["CustomerID", "Segment"]
]

# ------------------------------------------
# Merge Segment Information
# ------------------------------------------

clv_df = pd.merge(
    clv_df,
    segments,
    on="CustomerID",
    how="left"
)

print("Merged Shape:", clv_df.shape)

# ------------------------------------------
# One-Hot Encoding
# ------------------------------------------

clv_df = pd.get_dummies(
    clv_df,
    columns=["Segment"],
    drop_first=True
)

# ------------------------------------------
# Features
# ------------------------------------------

feature_cols = [

    "Recency",
    "Frequency",
    "PastRevenue",
    "AvgOrderValue",
    "TotalQuantity",
    "AvgQuantity",
    "ActiveDays"

]

# Add Segment Columns

segment_cols = [

    col for col in clv_df.columns
    if col.startswith("Segment_")

]

feature_cols.extend(segment_cols)

X = clv_df[feature_cols]

# ------------------------------------------
# Target
# ------------------------------------------

y = clv_df["FutureRevenue_Log"]

# ------------------------------------------
# Train Test Split
# ------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ------------------------------------------
# Models
# ------------------------------------------

models = {

    "Linear Regression":
        LinearRegression(),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=500,
            max_depth=12,
            random_state=42
        ),

    "XGBoost":
        XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.03,
            random_state=42
        )
}

# ------------------------------------------
# Train & Evaluate
# ------------------------------------------

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        preds
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            preds
        )
    )

    r2 = r2_score(
        y_test,
        preds
    )

    results.append([
        name,
        mae,
        rmse,
        r2
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R2"
    ]
)

print("\nResults:")
print(
    results_df.sort_values(
        by="R2",
        ascending=False
    )
)

Merged Shape: (3317, 11)

Results:
               Model       MAE      RMSE        R2
1      Random Forest  0.931632  1.532725  0.786054
2            XGBoost  0.949381  1.542818  0.783227
0  Linear Regression  1.180745  2.871442  0.249112


In [3]:
import joblib
import os

# Create models folder if not exists
os.makedirs("../models", exist_ok=True)

# Train final model again
final_clv_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=12,
    random_state=42
)

final_clv_model.fit(X_train, y_train)

# Save model
joblib.dump(
    final_clv_model,
    "../models/clv_model.pkl"
)

print("CLV model saved successfully!")

CLV model saved successfully!
